<a href="https://colab.research.google.com/github/namitbattancoc-design/veritas-AI-hackathon-project/blob/main/Veritas_AI_project_Code_medplain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re, subprocess, sys
import pandas as pd
from PIL import Image, ImageOps
from google.colab import files

subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr", shell=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytesseract"])
import pytesseract

DISCLAIMER = "This project is a prototype and is not medical advice."

# name: (aliases, fallback_low, fallback_high, meaning, question for doctor)
KB = {
 "Hemoglobin": (["hemoglobin","haemoglobin","hgb","hb"], 12.0, 17.5, "carries oxygen in your red blood cells", "Could this be anaemia, and do I need iron or B12 tests?"),
 "WBC": (["wbc","tlc","total leucocyte count","white blood cell"], 4.0, 11.0, "infection-fighting white cells", "Could an infection or a medicine explain my white cell count?"),
 "Platelets": (["platelet","platelets","platelet count","plt"], 150, 450, "helps blood clot", "Should I watch for bruising/bleeding, and when should I repeat this?"),
 "Glucose": (["glucose","fasting blood sugar","fbs","blood sugar"], 70, 99, "blood sugar (fasting)", "Was this fasting? Do I need an HbA1c test?"),
 "HbA1c": (["hba1c","hb a1c","a1c"], 4.0, 5.6, "3-month average blood sugar", "What does this mean for my diabetes risk?"),
 "Total cholesterol": (["cholesterol","total cholesterol"], 0, 200, "fat in the blood", "What is my heart risk and should I change my diet?"),
 "LDL": (["ldl","ldl cholesterol"], 0, 100, "'bad' cholesterol", "What LDL target is right for me?"),
 "HDL": (["hdl","hdl cholesterol"], 40, None, "'good' cholesterol", "How can I raise my HDL?"),
 "Triglycerides": (["triglyceride","triglycerides"], 0, 150, "blood fats", "Was I fasting, and what diet changes help?"),
 "Creatinine": (["creatinine"], 0.6, 1.3, "kidney filter marker", "Do I need a kidney (eGFR/urine) test?"),
 "TSH": (["tsh"], 0.4, 4.0, "thyroid signal", "Do I need T3/T4 tests?"),
 "Vitamin D": (["vitamin d"], 30, 100, "bone/muscle health vitamin", "Do I need a supplement, and what dose?"),
 "SGPT/ALT": (["alt","sgpt"], 7, 56, "liver enzyme", "Could my medicines or weight explain this?"),
}
NUM = r"\d+(?:\.\d+)?"

def preprocess(img):
    g = ImageOps.autocontrast(ImageOps.grayscale(img))
    return g.resize((g.width * 2, g.height * 2))

def ocr(path):
    return pytesseract.image_to_string(preprocess(Image.open(path)), config="--psm 6")

def match(name):
    n = re.sub(r"[^a-z0-9]+", " ", name.lower())
    best, bl = None, 0
    for k, (al, *_) in KB.items():
        for a in al:
            if re.search(rf"\b{re.escape(a)}\b", n) and len(a) > bl:
                best, bl = k, len(a)
    return best

def parse(text):
    rows = []
    for line in text.splitlines():
        m = re.match(rf"^\s*([A-Za-z][A-Za-z0-9 ()/%.\-]*?)\s*[:|]?\s+({NUM})\s*(.*)$", line)
        if not m: continue
        name, val, rest = m.groups()
        k = match(name)
        if not k: continue
        lo = hi = None
        r = re.search(rf"({NUM})\s*(?:-|–|to)\s*({NUM})", rest)
        if r: lo, hi = float(r[1]), float(r[2])
        elif (lt := re.search(rf"<\s*=?\s*({NUM})", rest)): hi = float(lt[1])
        elif (gt := re.search(rf">\s*=?\s*({NUM})", rest)): lo = float(gt[1])
        rows.append(dict(test=k, value=float(val), lo=lo, hi=hi))
    return rows

def judge(r):
    lo, hi, v, src = r["lo"], r["hi"], r["value"], "report"
    if lo is None and hi is None:                      # fall back to standard range
        _, lo, hi, *_ = KB[r["test"]]; src = "standard"
        if r["test"] in ("WBC", "Platelets") and v > 1000: v /= 1000   # /cumm -> thousands
    status = "low" if lo is not None and v < lo else "high" if hi is not None and v > hi else "normal"
    edge = lo if status == "low" else hi
    far = status != "normal" and edge and abs(v - edge) / abs(edge) >= 0.3
    return dict(r, status=status, far=bool(far), source=src)

def report(rows):
    res = [judge(r) for r in rows]
    bad = [r for r in res if r["status"] != "normal"]
    print("=" * 55, "\n MEDPLAIN: YOUR REPORT IN PLAIN LANGUAGE\n", "=" * 55)
    print(f"Read {len(res)} results: {len(res)-len(bad)} normal, {len(bad)} outside range.")
    print("Outside the range does NOT automatically mean illness.\n")
    for r in bad:
        _, _, _, meaning, _ = KB[r["test"]]
        print(f"{'🔴' if r['far'] else '🟠'} {r['test']}: {r['value']:g} is {r['status'].upper()} ({meaning})")
    print("\nQuestions to ask your doctor:")
    qs = [KB[r["test"]][4] for r in bad] + ["Should I repeat these tests, and when?",
          "Could my medicines, supplements or diet affect these numbers?"]
    for i, q in enumerate(qs, 1): print(f"{i}. {q}")
    print("\n⚠️", DISCLAIMER)
    return res

# ---------- 1) Try it on one report ----------
up = files.upload()
for fn in up:
    text = ocr(fn)
    print(text[:500], "\n")
    report(parse(text))

# ---------- 2) REAL audit: upload 5-10 synthetic report images ----------
# Fill TRUTH by hand from each image BEFORE running: {filename: {test: "low"/"high"/"normal"}}
TRUTH = {
    # "report1.png": {"Hemoglobin": "low", "Glucose": "normal", "Total cholesterol": "high"},
}
tp = fn_ = fp = tn = unread = 0
for fname, truth in TRUTH.items():
    got = {r["test"]: judge(r)["status"] for r in parse(ocr(fname))}
    for test, true_status in truth.items():
        if test not in got: unread += 1; continue
        t_abn, p_abn = true_status != "normal", got[test] != "normal"
        tp += t_abn and p_abn; fn_ += t_abn and not p_abn
        fp += (not t_abn) and p_abn; tn += (not t_abn) and not p_abn
if TRUTH:
    print(pd.DataFrame([dict(caught=tp, missed_abnormal=fn_, false_alarms=fp, correct_normal=tn, not_read=unread)]))
    print("Baseline 'everything normal' would miss ALL abnormal values.")
else:
    print("Fill TRUTH first, then re-run to get your real audit numbers.")

Saving IMG_62ac6896-e44a-428e-8764-e99584c60999~2.jpg to IMG_62ac6896-e44a-428e-8764-e99584c60999~2.jpg
HAEMATOLOGY
Test Name Value Unit Reference Range
COMPLETE HAEMOGRAM

CBC WITH P/S AND ESR,EDTA Blood

HAEMOGLOBIN 11.5 gm/ dl 13.0-18.0
by StS

TLC (Total Leucocyte Count) 13100 / cumm 4000-11000
by Automated Blectrical Resistance impedance

DIFFERENTIAL LEUCOCYTE COUNT

NEUTROPHIL 76.7 % 40-80
by Floweytometry

LYMPHOCYTES 17.7 % 20-40
by Flowcytometry

MONOCYTES 4.8 % 0-10
by Flowcytometry

EOSINOPHIL 0.6 % 0-6
by Flowcytometry

BASOPHILS 0.2 % 0-2
by Flowcytometry

ABSOLUTE NEUTROPHIL COUNT 1 

 MEDPLAIN: YOUR REPORT IN PLAIN LANGUAGE
Read 3 results: 1 normal, 2 outside range.
Outside the range does NOT automatically mean illness.

🟠 Hemoglobin: 11.5 is LOW (carries oxygen in your red blood cells)
🟠 WBC: 13100 is HIGH (infection-fighting white cells)

Questions to ask your doctor:
1. Could this be anaemia, and do I need iron or B12 tests?
2. Could an infection or a medicine explai